# SFT на Kaggle — подготовка и эксперименты
Включите T4 ×2 и Internet. Прикрепите датасет Spider + BIRD и разрешите доступ к Kaggle Secret `COMET_API_KEY`.
Выполните ячейки 1–5 по порядку. Они подготавливают среду и скачивают исходную модель. Обучение запускается отдельно через `run_trial` или `run_wave`.
Ревизия Base совпадает с измеренным baseline. Итоги и адаптер каждого завершённого обучения отправляются в Comet до оценки, после оценки туда добавляются metrics.json и ответы JSONL.


## 1. Пакеты
В новой сессии выполните целиком. После подготовки не переустанавливайте пакеты в ходе экспериментов.

In [ ]:
%pip install "vllm==0.31.0" "peft==0.21.2" accelerate comet-ml python-dotenv
%pip install --no-deps --index-url https://download.pytorch.org/whl/cu130 "torchaudio==2.11.0+cu130"
%pip uninstall -y torchao torchcodec

## 2. Репозиторий, пути и датасет
Если датасетов несколько, задайте DATA_DIR вручную в отмеченной строке.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "text2sql-grpo-sft"
RUN = WORK / "text2sql_runs" / "v1"
BASE = "Qwen/Qwen3.5-2B"
REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"

if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/dancravik/text2sql-grpo.git", str(REPO)], check=True)

DATA_DIR = None  # Например Path("/kaggle/input/datasets/your-name/text2sql")
if DATA_DIR is None:
    roots = sorted({
        p.parents[1] for p in INPUT.rglob("train_spider.json")
        if p.parent.name == "spider"
        and (p.parents[1] / "minidev/MINIDEV/mini_dev_sqlite.json").is_file()
    })
    if len(roots) == 1:
        DATA_DIR = roots[0]
    elif roots:
        raise ValueError(f"Укажите DATA_DIR вручную, найдены {roots}")
    else:
        DATA_DIR = WORK / "text2sql_data"
        if not (DATA_DIR / "spider/train_spider.json").is_file():
            archives = list(INPUT.rglob("text2sql-data.zip"))
            assert len(archives) == 1, f"Не найден датасет: {archives}"
            with ZipFile(archives[0]) as archive:
                archive.extractall(DATA_DIR)

RUN.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("DATA_DIR:", DATA_DIR)
print("Свободно GiB:", round(shutil.disk_usage(WORK).free / 1024**3, 2))
subprocess.run([sys.executable, "scripts/freeze_splits.py", "--verify"], cwd=REPO, check=True)

## 3. Comet и проверка GPU
Workspace ниже задан по предыдущим экспериментам. Измените его, если нужен другой проект Comet. Ключ извлекается из Kaggle Secrets и не выводится.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")
os.environ["COMET_WORKSPACE"] = "danchenko-rtm"
os.environ["COMET_PROJECT_NAME"] = "text2sql-grpo"

gpu_ids = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"], text=True
).split()
assert gpu_ids, "Включите GPU в настройках Kaggle"
print("GPU:", gpu_ids)

# Полный forward/backward свёртки проверяет cuDNN до загрузки модели.
subprocess.run([sys.executable, "-u", "-c", r'''
import importlib.metadata
import torch
from transformers import Trainer
from peft import LoraConfig
from vllm import LLM
for pkg in ("torch", "transformers", "peft", "accelerate", "vllm", "comet-ml"):
    print(pkg, importlib.metadata.version(pkg))
x = torch.randn(1, 6144, 512, device="cuda", dtype=torch.float16, requires_grad=True)
w = torch.randn(6144, 1, 4, device="cuda", dtype=torch.float16)
torch.nn.functional.conv1d(x, w, groups=6144, padding=3).float().mean().backward()
torch.cuda.synchronize()
print("GPU и FP16 conv1d forward/backward OK")
'''], check=True)

# Проверка реального подключения Comet до обучения.
subprocess.run([sys.executable, "-u", "-c", r'''
from comet_tracking import start_experiment
experiment = start_experiment(name="sft_environment_check")
try:
    experiment.log_metric("environment_ready", 1)
finally:
    experiment.end()
'''], env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)

## 4. Скачать Base с фиксированной ревизией
Этот шаг заполняет кэш Hugging Face для обоих GPU. Веса при каждом обучении загружаются из этого кэша. Повторная оценка Base не нужна.

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(BASE, revision=REVISION)
print("Модель в кэше:", model_path)
print("Свободно GiB:", round(shutil.disk_usage(WORK).free / 1024**3, 2))

## 5. Функции для следующих экспериментов
Микробатч 1 и накопление 8 уже проверены на старой T4. В коротких пилотах отключены промежуточные checkpoint-файлы. После обучения ZIP с адаптером и run.json отправляется в Assets того же эксперимента Comet. После оценки обновлённый ZIP включает полный JSON и JSONL. Затем удаляется только восстановимая копия merged. Для новой среды сначала выполните smoke через run_trial(smoke=True). При ошибке ячейка сохраняет остальные успешные результаты и показывает, какой запуск упал.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

# Новая группа для каждого повторения подбора, чтобы не смешивать результаты.
GROUP = "sft_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def run_script(script, *args, env=None):
    subprocess.run(
        [sys.executable, "-u", str(REPO / "scripts" / script), *map(str, args)],
        cwd=REPO, env=env, check=True,
    )

# Отдельный процесс для SDK позволяет безопасно архивировать параллельные запуски.
BACKUP_CODE = r'''
import json, sys
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from comet_tracking import start_experiment
run = Path(sys.argv[1])
evaluation = Path(sys.argv[2])
report = json.loads((run / "run.json").read_text())
archive = run / "artifacts.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    z.write(run / "run.json", "run.json")
    for path in sorted((run / "adapter").rglob("*")):
        if path.is_file():
            z.write(path, path.relative_to(run))
    if evaluation.exists():
        for path in sorted(evaluation.glob("*.json*")):
            z.write(path, "evaluation/" + path.name)
    z.write(Path(sys.argv[3]) / "splits/v1.json", "splits/v1.json")
experiment = start_experiment(key=report["comet_experiment_key"])
try:
    experiment.log_asset(str(archive), file_name="sft_artifacts.zip", overwrite=True)
finally:
    experiment.end()
print("Архив отправлен в Comet:", archive)
'''

def backup_trial(path, evaluation, env):
    subprocess.run(
        [sys.executable, "-u", "-c", BACKUP_CODE, str(path), str(evaluation), str(REPO)],
        env={**env, "PYTHONPATH": str(REPO / "src")}, check=True,
    )


def run_trial(label, gpu=0, lr=1e-4, warmup=0.03, rank=16,
              alpha=32, num_examples=1024, epochs=1, smoke=False):
    name = f"{GROUP}_{label}"
    path = RUN / name
    evaluation = RUN / "eval" / "validation_fast" / name
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    if path.exists():
        raise FileExistsError(f"Выберите новое label, каталог уже существует: {path}")
    # Две копии merged одновременно требуют около 8.3 GiB, плюс адаптеры и временные файлы.
    if shutil.disk_usage(WORK).free < 7 * 1024**3:
        raise RuntimeError("Свободно менее 7 GiB. Освободите место перед запуском.")
    duration = ("--smoke",) if smoke else ("--num-train-examples", num_examples, "--epochs", epochs)
    run_script(
        "train_sft.py", "--model", BASE, "--revision", REVISION,
        "--out-dir", path, "--seed", 42, "--comet",
        "--micro-batch-size", 1, "--grad-accum", 8,
        "--learning-rate", lr, "--warmup-ratio", warmup,
        "--lora-r", rank, "--lora-alpha", alpha,
        "--logging-steps", 5, "--save-strategy", "no", *duration, env=env,
    )
    # Сохранение обученного адаптера в облако до инициализации vLLM.
    backup_trial(path, evaluation, env)
    extra = ("--smoke",) if smoke else ()
    run_script(
        "evaluate.py", "--model", path / "merged", "--name", name,
        "--datasets", "validation_fast", "--out-dir", evaluation.parent,
        "--comet-run", path / "run.json",
        "--n", 1, "--temperature", 0, "--enable-thinking", "off",
        "--max-tokens", 1024, "--max-model-len", 4096, "--seed", 42,
        *extra, env=env,
    )
    backup_trial(path, evaluation, env)
    # Адаптер, отчёты и ZIP остаются локально и отправлены в Comet.
    shutil.rmtree(path / "merged")
    metrics = json.loads((evaluation / "metrics.json").read_text())["datasets"]["validation_fast"]
    result = {"name": name, "learning_rate": lr, "warmup_ratio": warmup,
              "lora_r": rank, "lora_alpha": alpha, "metrics": metrics}
    print(name, "accuracy", metrics["greedy_execution_accuracy"], flush=True)
    return result


def run_wave(configs):
    if len(configs) > len(gpu_ids):
        raise ValueError("В одной волне должен быть максимум один запуск на GPU")
    if shutil.disk_usage(WORK).free < 12 * 1024**3:
        raise RuntimeError("Для параллельной волны оставьте минимум 12 GiB свободного места.")
    successes, failures = [], []
    with ThreadPoolExecutor(max_workers=len(configs)) as pool:
        jobs = {pool.submit(run_trial, gpu=gpu, **cfg): cfg
                for cfg, gpu in zip(configs, gpu_ids)}
        for job in as_completed(jobs):
            try:
                successes.append(job.result())
            except Exception as error:
                failures.append({"config": jobs[job], "error": str(error)})
                print("Ошибка запуска:", jobs[job], str(error), flush=True)
    output = {"completed": successes, "failed": failures}
    summary = RUN / f"{GROUP}_wave_{configs[0]['label']}.json"
    summary.write_text(json.dumps(output, indent=2) + "\n")
    print("Сводка волны:", summary)
    if failures:
        raise RuntimeError("Некоторые запуски упали. Успешные отчёты сохранены, см. сводку волны.")
    return successes

print("Подготовка завершена. Группа экспериментов:", GROUP)

## Следующий запуск — выбирайте отдельно
Подготовка выше не запускает обучение. Для smoke выполните `smoke_result = run_trial("smoke", smoke=True)`. Для четырёх LR используйте следующую ячейку. Каждый пилот обучается одну эпоху на одних и тех же 1024 примерах и оценивается на одних и тех же 200 вопросах. Первая волна полностью заканчивается до начала второй. Не запускайте smoke и LR одновременно на одной карте. Оценки ранжируйте по execution accuracy, при близких результатах сравните лидеров на полной validation. Для последующего warmup используйте run_wave с выбранным lr и warmup=0.0/0.03. Для rank сравнивайте rank=8, alpha=16 и rank=16, alpha=32. При увеличении rank заново проверьте память. В этой версии выбирается итоговый чекпойнт каждой эпохи, автоматической проверки SQL внутри эпохи нет.

In [ ]:
# Запускайте эту ячейку только когда готовы начать подбор LR.
assert len(gpu_ids) >= 2, "Для двух параллельных запусков нужны две T4"
results = run_wave([
    {"label": "lr_1e4", "lr": 1e-4},
    {"label": "lr_5e5", "lr": 5e-5},
])
results += run_wave([
    {"label": "lr_2e4", "lr": 2e-4},
    {"label": "lr_2p5e5", "lr": 2.5e-5},
])
ranked = sorted(results, key=lambda r: (
    r["metrics"]["greedy_execution_accuracy"],
    r["metrics"]["greedy_execution_accuracy_allow_untagged"],
), reverse=True)
summary = RUN / f"{GROUP}_lr_selection.json"
summary.write_text(json.dumps(ranked, indent=2) + "\n")
for row in ranked:
    print(row["learning_rate"], row["metrics"]["greedy_execution_accuracy"], row["name"])
print("Итоговая сводка:", summary)